# Layer 8 — Narration State

This checkpoint tracks the exact PDF page and character offset through pause and resume. Audio synthesis leaves the cursor untouched; the notebook advances it only after treating a segment as fully played. The later interruption workflow can build on this state.

In [1]:
from pathlib import Path
import sys
import wave

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'src').is_dir():
    raise RuntimeError('Run this notebook from the ConversIQ project or notebooks directory.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from IPython.display import Audio, display
from src.extraction import extract_pdf
from src.narrator import (
    advance_narration, pause_narration, remaining_page_text,
    resume_narration, start_narration, synthesize_narration_segment,
)

pdf_path = PROJECT_ROOT / 'data' / 'sample_docs' / 'sample.pdf'
voice_path = PROJECT_ROOT / 'cache' / 'voices' / 'en_US-lessac-medium.onnx'
output_path = PROJECT_ROOT / 'cache' / 'tts' / 'layer8_narration.wav'
if not pdf_path.is_file():
    raise FileNotFoundError(f'Place a PDF at {pdf_path}, or change pdf_path.')
document = extract_pdf(pdf_path)
print(f'Document: {document.filename}; pages: {document.num_pages}')

Document: sample.pdf; pages: 41


## Position, pause, and resume

Advance by an exact number of characters to simulate text that has already played. Pausing and resuming must preserve both cursor coordinates.

In [2]:
state = start_narration(document)
if state.is_complete:
    raise RuntimeError('This PDF has no extractable text to narrate.')

already_played = min(40, len(remaining_page_text(state)) - 1)
state = advance_narration(state, already_played)
position_before_pause = (state.page_number, state.text_offset)
paused = pause_narration(state)
resumed = resume_narration(paused)
assert paused.is_paused and not resumed.is_paused
assert (resumed.page_number, resumed.text_offset) == position_before_pause
assert remaining_page_text(resumed) == remaining_page_text(state)

print(f'Already played: {already_played} characters')
print(f'Paused at page {paused.page_number}, character {paused.text_offset}')
print(f'Resumed at page {resumed.page_number}, character {resumed.text_offset}')
print(f'Remaining text on this page: {len(remaining_page_text(resumed))} characters')

Already played: 40 characters
Paused at page 1, character 40
Resumed at page 1, character 40
Remaining text on this page: 412 characters


## Narration audio checkpoint

Synthesize a short group of complete sentences (up to the character limit) using the Layer 7 Piper API. The continuous session loads the Piper voice once, avoiding a model reload between groups. Initials such as `J. K. Rowling` and titles such as `Mr.` stay in the same group; PDF whitespace is smoothed only for speech, while the returned segment retains exact source characters for cursor tracking. The state remains at the same position until the segment is marked fully played by `advance_narration()`. If interrupted, that segment replays from its start; no unheard text is skipped.

In [3]:
if not voice_path.is_file():
    raise FileNotFoundError(
        'Download the Piper voice first: uv run python -m piper.download_voices '
        '--download-dir cache/voices en_US-lessac-medium'
    )

segment_text, audio_path = synthesize_narration_segment(
    resumed, voice_path, output_path, max_characters=140
)
assert 0 < len(segment_text) <= 140
assert remaining_page_text(resumed).startswith(segment_text)
assert (resumed.page_number, resumed.text_offset) == position_before_pause
after_playback = advance_narration(resumed, len(segment_text))
with wave.open(str(audio_path), 'rb') as wav_file:
    duration = wav_file.getnframes() / wav_file.getframerate()
assert duration > 0

print(f'Synthesized segment ({len(segment_text)} characters): {segment_text!r}')
print(f'Audio duration: {duration:.2f} s')
print(f'Cursor before playback: page {resumed.page_number}, character {resumed.text_offset}')
print(f'Cursor after playback: page {after_playback.page_number}, character {after_playback.text_offset}')
display(Audio(filename=str(audio_path)))

Synthesized: 140 characters
Audio duration: 10.58 s
Cursor before playback: page 1, character 40
Cursor after playback: page 1, character 180


# Layer 9 — Spoken Interruption and Resume

Record a question about `sample.pdf` as `data/sample_docs/sample_audio.wav` and build that PDF's PageIndex tree first. This manual cell sends the recording and selected PDF source text to Groq, plays a speech-friendly copy of the answer (without Markdown or citation markers), and then resumes at the exact saved text cursor. The displayed answer remains unchanged. Run it once when ready.

In [4]:
from src.config import CACHE_DIR
from src.extraction import generate_doc_id
from src.graph import get_interrupt_graph

question_audio_path = PROJECT_ROOT / 'data' / 'sample_docs' / 'sample_audio.wav'
if not question_audio_path.is_file():
    raise FileNotFoundError(f'Record a spoken question at: {question_audio_path}')
doc_id = generate_doc_id(pdf_path)
tree_cache_path = CACHE_DIR / f'{doc_id}.json'
if not tree_cache_path.is_file():
    raise FileNotFoundError(f'Build the PageIndex tree for this PDF first: {tree_cache_path}')

position_before_question = (after_playback.page_number, after_playback.text_offset)
answer_wav_path = PROJECT_ROOT / 'cache' / 'tts' / 'layer9_answer.wav'
workflow = get_interrupt_graph()
final_state = workflow.invoke({
    'narration_state': after_playback,
    'doc_id': doc_id,
    'file_path': pdf_path,
    'audio_path': question_audio_path,
    'voice_path': voice_path,
    'answer_audio_path': answer_wav_path,
})
resumed_state = final_state['narration_state']
assert not resumed_state.is_paused
assert (resumed_state.page_number, resumed_state.text_offset) == position_before_question

print(f'Question transcript: {final_state.get("query", "") or "<no speech recognized>"}')
print(f'Selected pages: {list(final_state.get("relevant_pages", ()))}')
print(f'Answer: {final_state.get("answer", "<none>")}')
print(f'Answer audio played: {final_state.get("answer_audio_played", False)}')
print(f'Resumed at page {resumed_state.page_number}, character {resumed_state.text_offset}')
if final_state.get('error'):
    print(f'Status: {final_state["error"]}')

Question transcript: Portkey is a production ready AI control platform and gateway that connects application to over 1600 large language models using a single unified API.
Selected pages: [4, 5, 6]
Answer: The provided source (pages 4‑6) does not contain any information about a product called **Portkey**, nor does it mention a platform that connects applications to over 1,600 large language models via a unified API. Therefore, I cannot confirm or elaborate on that statement from the given text.
Answer audio played: True
Resumed at page 1, character 180


# Layer 10 — One-cycle application integration

The thin `main.py` entry point narrates one segment, handles a prerecorded or newly recorded spoken question, plays its answer, and continues narration from the saved cursor. Set the flag below to `True` only when you want another Groq/Piper/audio run. For microphone mode, use `--record-seconds 5` instead of `--question-audio`. Add `--interrupt-on-enter` for keyboard interruption or `--interrupt-on-speech` for opt-in Silero VAD. An interrupted segment replays from its start after the answer. Speech mode needs the local `cache/vad/silero_vad.onnx` model and headphones to reduce speaker bleed.

For the full local interface, run `uv run python app.py` from the project root, choose a PDF and the microphone that actually hears your voice, then select **Start narration**. Speak during playback to interrupt and ask a question. The window shows recognized questions and grounded answers; audio plays through this computer's speakers. Headphones reduce the chance of narrator audio triggering VAD, but are not required. **Stop** ends the session without advancing an unfinished segment. The interface does not call Groq until a question is asked.

In [5]:
RUN_LAYER10_MANUAL = False
if RUN_LAYER10_MANUAL:
    from main import run_once

    integrated = run_once(
        pdf_path,
        PROJECT_ROOT / 'data' / 'sample_docs' / 'sample_audio.wav',
        voice_path,
        PROJECT_ROOT / 'cache' / 'tts',
    )
    assert not integrated['narration_state'].is_paused
    print(f'Question: {integrated.get("query", "")}')
    print(f'Answer: {integrated.get("answer", "<none>")}')
    print(f'Resumed cursor: {integrated["narration_state"].page_number}, {integrated["narration_state"].text_offset}')

## Optional microphone checkpoint

Set the flag below to `True` to record three seconds from the default microphone and listen locally. This cell does not call Groq or change the narration state.

In [6]:
RUN_MICROPHONE_CHECK = False
if RUN_MICROPHONE_CHECK:
    from src.audio import record_microphone

    microphone_wav = record_microphone(
        PROJECT_ROOT / 'cache' / 'tts' / 'microphone_check.wav', 3
    )
    with wave.open(str(microphone_wav), 'rb') as recording:
        assert recording.getnchannels() == 1
        assert recording.getsampwidth() == 2
        print(f'Microphone recording: {recording.getnframes() / recording.getframerate():.2f} seconds')
    display(Audio(filename=str(microphone_wav)))

## Optional interruptible playback checkpoint

Set the flag below to `True` to play the existing narration WAV and stop it after about one second. No Groq call occurs. The narration cursor stays untouched because the full segment was not heard.

In [ ]:
RUN_INTERRUPTIBLE_PLAYBACK_CHECK = False
if RUN_INTERRUPTIBLE_PLAYBACK_CHECK:
    from time import perf_counter
    from src.audio import play_wav_interruptible

    cursor_before = (resumed.page_number, resumed.text_offset)
    playback_started = perf_counter()
    completed = play_wav_interruptible(
        output_path, lambda: perf_counter() - playback_started >= 1.0
    )
    assert not completed
    assert (resumed.page_number, resumed.text_offset) == cursor_before
    print(f'Playback stopped early; cursor remains at {cursor_before}')

## Optional Silero speech-interruption checkpoint

The ONNX model comes from the [official Silero repository](https://github.com/snakers4/silero-vad/blob/master/src/silero_vad/data/silero_vad.onnx) and belongs at `cache/vad/silero_vad.onnx`. Wear headphones: audio from speakers can be heard by the microphone and falsely trigger VAD. Enable the cell only when ready to play narration and speak over it. It records the question locally and does **not** call Groq.

In [ ]:
RUN_SPEECH_INTERRUPT_CHECK = False
if RUN_SPEECH_INTERRUPT_CHECK:
    from src.audio import monitor_microphone, play_wav_interruptible
    from src.vad import SileroVad

    vad_model = PROJECT_ROOT / 'cache' / 'vad' / 'silero_vad.onnx'
    detector = SileroVad(vad_model)
    silence_probability = detector.speech_probability(bytes(1024))
    print(f'Silence probability: {silence_probability:.3f}')
    assert 0 <= silence_probability <= 1

    cursor_before = (resumed.page_number, resumed.text_offset)
    with monitor_microphone(vad_model) as microphone:
        completed = play_wav_interruptible(
            output_path, microphone.speech_detected
        )
        question_wav = (
            microphone.capture_question(
                PROJECT_ROOT / 'cache' / 'tts' / 'speech_question.wav', 3
            )
            if not completed else None
        )
    assert (resumed.page_number, resumed.text_offset) == cursor_before
    print('Speech interrupted narration' if question_wav else 'No speech detected')
    if question_wav:
        display(Audio(filename=str(question_wav)))

## Optional continuous narration checkpoint

This runs the same PDF across up to three narration attempts without re-extracting, rebuilding the tree, or recompiling the graph between segments. Silence advances to the next segment; detected speech runs the answer workflow, then narration replays the existing interrupted WAV rather than synthesizing it again. The microphone is opened only during narration and question capture, so the Windows microphone icon may disappear between chunks. It uses the microphone, speakers, Piper, and possibly Groq. Wear headphones and enable it only for a manual test.

In [ ]:
RUN_CONTINUOUS_SESSION_CHECK = False
if RUN_CONTINUOUS_SESSION_CHECK:
    from src.session import run_continuous_session

    session_result = run_continuous_session(
        pdf_path, voice_path,
        PROJECT_ROOT / 'cache' / 'vad' / 'silero_vad.onnx',
        PROJECT_ROOT / 'cache' / 'tts',
        record_seconds=5, segment_characters=140, max_segments=3,
    )
    cursor = session_result.narration_state
    print(f'Segments attempted: {session_result.segments_attempted}')
    print(f'Speech interruptions: {session_result.interruptions}')
    print(f'Complete: {cursor.is_complete}')
    print(f'Cursor: page {cursor.page_number}, character {cursor.text_offset}')